In [ ]:
import torch, torch.nn as nn, urllib.request, time
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# 1. Load data (Tiny Shakespeare) and build character vocabulary
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url).read().decode()
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], device=device)
print(f"{len(text):,} characters, {len(chars)} unique")

In [ ]:
# 2. The RNN model: Embedding -> 2-layer LSTM -> Linear
class CharRNN(nn.Module):
    def __init__(self, vocab, emb=128, hidden=512, layers=2):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb)
        self.rnn = nn.LSTM(emb, hidden, layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden, vocab)
    def forward(self, x, hid=None):
        out, hid = self.rnn(self.emb(x), hid)
        return self.fc(out), hid

model = CharRNN(len(chars)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 3. Train: predict the next character for every position
B, T, STEPS = 128, 128, 2000
def get_batch():
    ix = torch.randint(len(data) - T - 1, (B,))
    x = torch.stack([data[i:i+T] for i in ix])
    y = torch.stack([data[i+1:i+T+1] for i in ix])
    return x, y

losses, start = [], time.time()
for step in range(1, STEPS + 1):
    x, y = get_batch()
    logits, _ = model(x)
    loss = loss_fn(logits.view(-1, len(chars)), y.view(-1))
    opt.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); losses.append(loss.item())
    if step % 250 == 0:
        print(f"step {step}/{STEPS}  loss {loss.item():.3f}  ({time.time()-start:.0f}s)")

In [ ]:
# 4. Generate text, with a "creativity" (temperature) knob
@torch.no_grad()
def generate(prompt="ROMEO:\n", n=400, temp=0.8):
    model.eval()
    x = torch.tensor([[stoi[c] for c in prompt]], device=device)
    logits, hid = model(x)
    out = prompt
    for _ in range(n):
        probs = torch.softmax(logits[:, -1] / temp, dim=-1)
        nxt = torch.multinomial(probs, 1)
        out += itos[nxt.item()]
        logits, hid = model(nxt, hid)
    return out

for t in [0.5, 1.0]:
    print(f"\n===== temperature {t} =====")
    print(generate("ROMEO:\n", temp=t))

In [ ]:
# 5. Try your own prompt + loss curve
import matplotlib.pyplot as plt
plt.plot(losses); plt.xlabel("step"); plt.ylabel("loss"); plt.title("Training loss"); plt.show()

print(generate("JULIET:\nO, my dear friend,", n=500, temp=0.7))